In [1]:
from ipyleaflet import Map, basemaps, TileLayer, GeoJSON, WidgetControl
from ipywidgets import HTML
import json
import urllib.request
# import subsample_pyramid
import xarray as xr
import geopandas as gpd

In [16]:
hydrobasins_lev05_url = 'https://raw.githubusercontent.com/blackteacatsu/spring_2024_envs_research_amazon_ldas/main/resources/hybas_sa_lev05_areaofstudy.geojson'

with urllib.request.urlopen(hydrobasins_lev05_url) as url:
    geojson_data = json.loads(url.read().decode())

polygon_layer = GeoJSON(
    data=geojson_data,
    style={
        'color': 'grey',
        'weight': 0.6,
        'fillOpacity': 0,
        'opacity': 1
    },
    hover_style={
        'color': 'grey',
        'weight': 0,
        'fillOpacity': 0.4
    },
    name='HydroBasins'
)

#m.add_layer(polygon_layer)

# info_html = HTML(value='<b>Amazon HydroViewer</b><br>Select variable and time')
# info_control = WidgetControl(widget=info_html, position='topright')
# m.add_control(info_control)

In [18]:
center = [-7.5, -65.5]
zoom = 4

m = Map(
    basemap=basemaps.CartoDB.Voyager, 
    center=center, 
    zoom=zoom,
    scroll_wheel_zoom=True
)

m.add_layer(polygon_layer)


# Add tile server layer to ipyleaflet
import requests

# Configuration
TILE_SERVER_URL = "http://localhost:5000"
variable = "Rainf_tavg"
time_idx = 0
category = 0
profile = 0
vmin = 40  # Minimum value for colormap
vmax = 100  # Maximum value for colormap

# Clear cache first to ensure fresh tiles
try:
    requests.get(f"{TILE_SERVER_URL}/cache/clear")
    print("✓ Cache cleared")
except:
    print("⚠ Could not clear cache")

tile_url = f"{TILE_SERVER_URL}/tiles/{variable}/{time_idx}/{category}/{{z}}/{{x}}/{{y}}.png?colormap=Reds&profile={profile}&mode=global&tms=true&vmin={vmin}&vmax={vmax}"


# Add tile layer (tms=False is default for ipyleaflet - uses XYZ coordinates)
data_layer = TileLayer(
    url=tile_url,
    name=f"{variable} - Category {category}",
    opacity=1,
    attribution='HydroViewer',
    min_native_zoom = 4,
    max_native_zoom = 9
)

m.add_layer(data_layer)

# Build tile URL with mode=global and tms=true
# for category in categories:
#     if category == 0:
#         tile_url = f"{TILE_SERVER_URL}/tiles/{variable}/{time_idx}/{category}/{{z}}/{{x}}/{{y}}.png?colormap=Reds&profile={profile}&mode=global&tms=true&vmin={vmin}&vmax={vmax}"


#         # Add tile layer (tms=False is default for ipyleaflet - uses XYZ coordinates)
#         data_layer = TileLayer(
#             url=tile_url,
#             name=f"{variable} - Category {category}",
#             opacity=0.6,
#             attribution='HydroViewer',
#             min_native_zoom = 4,
#             max_native_zoom = 9
#         )

#         m.add_layer(data_layer)

#     if category == 1:
#         tile_url = f"{TILE_SERVER_URL}/tiles/{variable}/{time_idx}/{category}/{{z}}/{{x}}/{{y}}.png?colormap=Greys&profile={profile}&mode=global&tms=true&vmin={vmin}&vmax={vmax}"


#         # Add tile layer (tms=False is default for ipyleaflet - uses XYZ coordinates)
#         data_layer = TileLayer(
#             url=tile_url,
#             name=f"{variable} - Category {category}",
#             opacity=0.6,
#             attribution='HydroViewer',
#             min_native_zoom = 4,
#             max_native_zoom = 9
#         )

#         m.add_layer(data_layer)
        
#     if category == 2:
#         tile_url = f"{TILE_SERVER_URL}/tiles/{variable}/{time_idx}/{category}/{{z}}/{{x}}/{{y}}.png?colormap=Blues&profile={profile}&mode=global&tms=true&vmin={vmin}&vmax={vmax}"

#         # Add tile layer (tms=False is default for ipyleaflet - uses XYZ coordinates)
#         data_layer = TileLayer(
#             url=tile_url,
#             name=f"{variable} - Category {category}",
#             opacity=0.8,
#             attribution='HydroViewer',
#             min_native_zoom = 4,
#             max_native_zoom = 9
#         )

#         m.add_layer(data_layer)

# print(f"✓ Added tile layer: {variable}")
# print(f"  vmin={vmin}, vmax={vmax}")

hover_info = HTML(value='<b>Hover over a basin</b>')
hover_control = WidgetControl(widget=hover_info, position='topright')
m.add_control(hover_control)

# Define hover callback to show pfaf_id
def on_hover(event, feature, **kwargs):
    pfaf_id = feature['properties'].get('PFAF_ID', 'N/A')
    hover_info.value = f'<b>Regional PFAF ID:</b> {pfaf_id}'

def on_click(event, feature, **kwargs):
    pfaf_id = feature['properties'].get('PFAF_ID', 'N/A')
    print(pfaf_id)
    return pfaf_id

polygon_layer.on_hover(on_hover)
polygon_layer.on_click(on_click)

m

✓ Cache cleared


Map(center=[-7.5, -65.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib as mpl
# import numpy as np
# from io import BytesIO
# import base64

# category_names = ["BELOW NORMAL", "NEAR NORMAL", "ABOVE NORMAL"]
# category_colors = ["Reds", "Greys", "Blues"]  # Colormaps for each category

# def create_category_colorbar(vmin, vmax, category_idx):
#     """Create colorbar for a specific category matching Shiny app style"""
#     fig, ax = plt.subplots(figsize=(3, 0.4))
#     fig.patch.set_alpha(0)
    
#     # Get colormap for this category
#     cmap = plt.get_cmap(category_colors[category_idx])
#     norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    
#     # Create horizontal colorbar
#     cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, 
#                                     orientation='horizontal')
    
#     # Style to match Shiny app
#     cb.ax.tick_params(labelsize=8, colors='#333', length=3)
#     cb.outline.set_linewidth(0.5)
#     cb.outline.set_edgecolor('#ccc')
    
#     # Set ticks
#     ticks = np.linspace(vmin, vmax, 6)
#     cb.set_ticks(ticks)
#     cb.set_ticklabels([f'{int(t)}' for t in ticks])
    
#     ax.set_facecolor('none')
    
#     # Save to base64
#     buf = BytesIO()
#     plt.savefig(buf, format='png', bbox_inches='tight', 
#                 facecolor='none', edgecolor='none', dpi=100, transparent=True)
#     plt.close(fig)
#     buf.seek(0)
    
#     img_base64 = base64.b64encode(buf.read()).decode('utf-8')
#     return f'data:image/png;base64,{img_base64}'

# # Create colorbar HTML matching Shiny app layout
# colorbar_images = []
# for i in range(3):
#     if i == category:
#         # Highlighted category (current selection)
#         colorbar_images.append(create_category_colorbar(vmin, vmax, i))
#     else:
#         # Grayed out categories
#         colorbar_images.append(None)

# # Build HTML matching Shiny app style
# colorbar_html_content = '''
# <div style="background-color: white; 
#             padding: 15px; 
#             border-radius: 5px;
#             box-shadow: 0 2px 6px rgba(0,0,0,0.2);
#             font-family: Arial, sans-serif;
#             min-width: 200px;">
# '''

# for i, cat_name in enumerate(category_names):
#     if i == category:
#         # Active category
#         colorbar_html_content += f'''
#         <div style="margin-bottom: 15px;">
#             <div style="font-size: 11px; 
#                         font-weight: bold; 
#                         color: #333; 
#                         margin-bottom: 5px;
#                         text-align: center;">
#                 {cat_name}
#             </div>
#             <img src="{colorbar_images[i]}" style="width: 100%; height: auto;">
#         </div>
#         '''
#     else:
#         # Inactive categories (grayed out)
#         colorbar_html_content += f'''
#         <div style="margin-bottom: 15px; opacity: 0.3;">
#             <div style="font-size: 11px; 
#                         color: #999; 
#                         margin-bottom: 5px;
#                         text-align: center;">
#                 {cat_name}
#             </div>
#             <div style="height: 40px; 
#                         background: linear-gradient(to right, #e0e0e0, #f5f5f5);
#                         border: 1px solid #ddd;
#                         border-radius: 3px;">
#             </div>
#         </div>
#         '''

# colorbar_html_content += '</div>'

# colorbar_widget = HTML(value=colorbar_html_content)

# # Add colorbar to map
# colorbar_control = WidgetControl(widget=colorbar_widget, position='topright')
# m.add_control(colorbar_control)

# print(f"✓ Added tile layer: {variable}")
# print(f"  vmin={vmin}, vmax={vmax}")
# #print(f"  colormap={colormap_name}")
# print(f"  Category: {category_names[category]}")
# print(f"  Colorbar added (Shiny app style)")

# m

✓ Added tile layer: Rainf_tavg
  vmin=40, vmax=100
  Category: ABOVE NORMAL
  Colorbar added (Shiny app style)


Map(bottom=2307.0, center=[-5.178482088522876, -65.12702512110076], controls=(ZoomControl(options=['position',…